In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/CleanSQL')
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuda


In [ ]:
# Create a temporary requirements file without vllm and transformers
!grep -vE '^(vllm|transformers)' requirements.txt > requirements_filtered.txt

# Install vllm
!pip install vllm==0.11.2

# Install remaining dependencies from the filtered list, ignoring already installed packages
!pip install -q -r requirements_filtered.txt --ignore-installed

# Clean up the temporary file
!rm requirements_filtered.txt

INFO: pip is looking at multiple versions of model-hosting-container-standards to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.3/370.3 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

^C


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 43.3 MB/s eta 0:00:00


In [ ]:
import json
import random
from pathlib import Path

input_file = Path('data/cleansql_500.jsonl')
train_file = Path('data/cleansql_500_train.jsonl')
val_file = Path('data/cleansql_500_val.jsonl')

examples = []
with input_file.open() as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

random.seed(42)
random.shuffle(examples)
split_idx = int(len(examples) * 0.8)

train_examples = examples[:split_idx]
val_examples = examples[split_idx:]

with train_file.open('w') as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

with val_file.open('w') as f:
    for ex in val_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f'Split {len(examples)} examples: {len(train_examples)} train, {len(val_examples)} val')

Split 500 examples: 400 train, 100 val


Fine-Tuning

In [ ]:
!python -m cleansql.llm.peft_trainer \
  --train-file data/cleansql_500_train.jsonl \
  --val-file data/cleansql_500_val.jsonl \
  --output-dir work/cleansql_500 \
  --base-model Qwen/Qwen2.5-Coder-7B-Instruct \
  --lora-r 8 \
  --lora-alpha 16 \
  --lora-dropout 0.05 \
  --learning-rate 5e-5 \
  --num-train-epochs 3 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 64 \
  --max-length 2048

2025-12-03 08:34:38.317976: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764750878.339358   17877 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764750878.345979   17877 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764750878.362879   17877 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764750878.362909   17877 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764750878.362912   17877 computation_placer.cc:177] computation placer alr

Restarting vLLM with Merged Model

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
lora_path = "work/cleansql_500"
merged_output = "work/cleansql_500_merged"

print('Loading base model...')
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base, lora_path)

print('Merging...')
merged = model.merge_and_unload()

print('Saving merged model and tokenizer...')
merged.save_pretrained(merged_output)
tokenizer.save_pretrained(merged_output)

print('✓ Done!')

Loading base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading LoRA adapter...
Merging...
Saving merged model and tokenizer...
✓ Done!


In [ ]:
# Clean up
import gc
for name in ["base", "model", "merged"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

Restarting vLLM

In [ ]:
# Stop existing vLLM
!pkill -f vllm || echo "No vLLM process found"
import time
time.sleep(5)

# Start vLLM with merged weights but base tokenizer
!python -m vllm.entrypoints.openai.api_server \
  --model work/cleansql_500_merged \
  --tokenizer Qwen/Qwen2.5-Coder-7B-Instruct \
  --trust-remote-code \
  --port 8000 \
  --gpu-memory-utilization 0.5 \
  --served-model-name qwen-coder \
  --tensor-parallel-size 1 \
  --dtype bfloat16 \
  > vllm_ft.log 2>&1 &

print("Waiting for vLLM to start...")
import requests
for i in range(15):
    time.sleep(10)
    try:
        response = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        print(f"✓ vLLM is ready! {response.json()}")
        break
    except Exception as e:
        print(f"  Still starting... ({i+1}/15)")

# Verify endpoint
try:
    test = requests.post(
        "http://127.0.0.1:8000/v1/chat/completions",
        json={"model": "qwen-coder", "messages": [{"role": "user", "content": "test"}]},
        timeout=30,
    )
    test.raise_for_status()
    print(f"✓ Chat endpoint working! Status: {test.status_code}")
    print(test.json())
except Exception as e:
    print(f"✗ Endpoint test failed: {e}")
    !sed -n '1,80p' vllm_ft.log

^C
Waiting for vLLM to start...
  Still starting... (1/15)
  Still starting... (2/15)
  Still starting... (3/15)
  Still starting... (4/15)
  Still starting... (5/15)
  Still starting... (6/15)
  Still starting... (7/15)
  Still starting... (8/15)
✓ vLLM is ready! {'object': 'list', 'data': [{'id': 'qwen-coder', 'object': 'model', 'created': 1764752313, 'owned_by': 'vllm', 'root': 'work/cleansql_500_merged', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-a095398b6a56481088e00b7b062f6274', 'object': 'model_permission', 'created': 1764752313, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}
✓ Chat endpoint working! Status: 200
{'id': 'chatcmpl-d6e0843e649e4c15bd7f416a87221d0d', 'object': 'chat.completion', 'created': 1764752316, 'model': 'qwen-coder', 'choices': [{'index': 0, 'message': {'role':

Run Stage 2 Evaluation

Evaluate the fine-tuned model:

In [ ]:
!pip install FlagEmbedding qdrant_client

In [ ]:
!python -m cleansql.eval.run_stage2 \
  --context rag_topk5 \
  --technique sc \
  --ids data/stage2_ids.jsonl \
  --db-root work/corrupted_databases \
  --clean-db-root data/spider_databases \
  --profile-dir data/profiles \
  --index-dir work/qdrant_index \
  --out results/finetuned_stage2_500.csv

2025-12-03 08:58:49.397181: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764752329.426938   24170 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764752329.433409   24170 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764752329.450046   24170 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764752329.450082   24170 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764752329.450085   24170 computation_placer.cc:177] computation placer alr

In [ ]:
!python analyze_failures_500.py \
  --ids data/stage2_ids.jsonl \
  --db-root work/corrupted_databases \
  --clean-db-root data/spider_databases \
  --profile-dir data/profiles \
  --index-dir work/qdrant_index \
  --out results/stage2_500_failures.csv \
  --technique sc \
  --rag-topk 5

2025-12-03 09:21:30.513050: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764753690.534161   29937 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764753690.540742   29937 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764753690.557269   29937 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764753690.557296   29937 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764753690.557299   29937 computation_placer.cc:177] computation placer alr

Compare Results

Compare fine-tuned model vs baseline:

In [ ]:
import pandas as pd

# Load results
baseline = pd.read_csv('results/stage2_rag_k5_sc.csv')
finetuned = pd.read_csv('results/finetuned_stage2_500.csv')

print('=' * 70)
print('COMPARISON: Baseline vs Fine-Tuned Model (500 Dataset)')
print('=' * 70)
print(f"{'Metric':<25} {'Baseline':<15} {'Fine-Tuned':<15} {'Change':<15}")
print('-' * 70)

metrics = ['dq_robust_1pct', 'dq_robust_5pct', 'mape', 'mae', 'literal_acc', 'dq_note_coverage']
comparison_data = []

for metric in metrics:
    base_val = baseline[metric].mean()
    fine_val = finetuned[metric].mean()

    if metric in ['mape', 'mae']:
        # Lower is better
        change = base_val - fine_val
        change_pct = (change / base_val * 100) if base_val != 0 else 0
    else:
        # Higher is better
        change = fine_val - base_val
        change_pct = (change / base_val * 100) if base_val != 0 else 0

    print(f"{metric:<25} {base_val:<15.4f} {fine_val:<15.4f} {change:+.4f} ({change_pct:+.1f}%)")

    # Store for saving
    comparison_data.append({
        'metric': metric,
        'baseline': base_val,
        'finetuned': fine_val,
        'change': change,
        'change_pct': change_pct
    })

print('=' * 70)

# SAVE COMPARISON TABLE
comparison_df = pd.DataFrame(comparison_data)
comparison_file = 'results/comparison_baseline_vs_finetuned_500.csv'
comparison_df.to_csv(comparison_file, index=False)

print(f'\n✓ Comparison table saved to: {comparison_file}')

# Summary table
print('\n' + '=' * 70)
print('SUMMARY TABLE')
print('=' * 70)
summary_data = {
    'Configuration': ['Baseline (Base Model)', 'Fine-Tuned (500 Dataset)'],
    'dq_robust_1pct': [
        f"{baseline['dq_robust_1pct'].mean():.1%}",
        f"{finetuned['dq_robust_1pct'].mean():.1%}"
    ],
    'dq_robust_5pct': [
        f"{baseline['dq_robust_5pct'].mean():.1%}",
        f"{finetuned['dq_robust_5pct'].mean():.1%}"
    ],
    'MAPE': [
        f"{baseline['mape'].mean():.2f}",
        f"{finetuned['mape'].mean():.2f}"
    ],
    'MAE': [
        f"{baseline['mae'].mean():.2f}",
        f"{finetuned['mae'].mean():.2f}"
    ],
    'Literal Acc': [
        f"{baseline['literal_acc'].mean():.1%}",
        f"{finetuned['literal_acc'].mean():.1%}"
    ],
    'Note Coverage': [
        f"{baseline['dq_note_coverage'].mean():.1%}",
        f"{finetuned['dq_note_coverage'].mean():.1%}"
    ],
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# SAVE SUMMARY TABLE
summary_file = 'results/summary_baseline_vs_finetuned_500.csv'
summary_df.to_csv(summary_file, index=False)

print(f'\n✓ Summary table saved to: {summary_file}')
print('=' * 70)
print('EVALUATION COMPLETE!')
print('=' * 70)

COMPARISON: Baseline vs Fine-Tuned Model (500 Dataset)
Metric                    Baseline        Fine-Tuned      Change         
----------------------------------------------------------------------
dq_robust_1pct            0.7814          0.7260          -0.0553 (-7.1%)
dq_robust_5pct            0.7974          0.7295          -0.0680 (-8.5%)
mape                      10.2168         0.3272          +9.8896 (+96.8%)
mae                       134.3848        271.5534        -137.1686 (-102.1%)
literal_acc               0.3706          0.4247          +0.0541 (+14.6%)
dq_note_coverage          1.0000          1.0000          +0.0000 (+0.0%)

✓ Comparison table saved to: results/comparison_baseline_vs_finetuned_500.csv

SUMMARY TABLE
           Configuration dq_robust_1pct dq_robust_5pct  MAPE    MAE Literal Acc Note Coverage
   Baseline (Base Model)          78.1%          79.7% 10.22 134.38       37.1%        100.0%
Fine-Tuned (500 Dataset)          72.6%          72.9%  0.33 271.55 